# 08 — Transformer applications: BERT, GPT, T5/BART, prompting, pipelines, fine-tuning templates

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Transformer families:

| Family | Architecture | Best for |
|---|---|---|
| BERT | encoder-only | understanding, classification, NER, extractive QA |
| GPT | decoder-only | generation, chat, agents, next-token prediction |
| T5/BART | encoder-decoder | translation, summarisation, text-to-text transformation |

This notebook uses Hugging Face-style APIs. The package may need installation in a fresh environment.

In [ ]:
# Uncomment in a fresh environment.
# %pip install transformers datasets evaluate accelerate

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

## Pipelines for fast prototypes

Pipelines are useful for demos and baselines. For production, you usually want explicit tokenizers, models, batching, monitoring, and evals.

In [ ]:
try:
    from transformers import pipeline
    sentiment_pipe = pipeline("sentiment-analysis")
    print(sentiment_pipe("The dashboard saves me hours every week."))
except Exception as e:
    print("Transformers pipeline unavailable or model download blocked:")
    print(e)

## BERT-style classification template

Use this when you have labelled examples and need consistent classification behavior.

Replace `model_name` and `csv_path` with your own model and data.

In [ ]:
# Template: fine-tune a BERT-style model for sequence classification.
# This is intentionally written as reusable project code.

bert_finetune_template = r"""
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

model_name = "distilbert-base-uncased"
dataset = load_dataset("csv", data_files={"train": "train.csv", "validation": "valid.csv"})
label_names = sorted(set(dataset["train"]["label"]))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)
    enc["labels"] = [label2id[x] for x in batch["label"]]
    return enc

tokenized = dataset.map(preprocess, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": metric.compute(predictions=preds, references=labels, average="macro")["f1"]}

args = TrainingArguments(
    output_dir="outputs/bert_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"], tokenizer=tokenizer, compute_metrics=compute_metrics)
trainer.train()
trainer.evaluate()
trainer.save_model("models/bert_classifier")
"""
print(bert_finetune_template[:1200])

## BERT token classification template for NER

Use this for BIO-tagged entity extraction.

In [ ]:
bert_ner_template = r"""
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

def tokenize_and_align_labels(example):
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    labels = []
    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous_word_id:
            labels.append(label2id[example["ner_tags"][word_id]])
        else:
            # Common choice: ignore non-first subwords
            labels.append(-100)
        previous_word_id = word_id
    tokenized["labels"] = labels
    return tokenized

model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), id2label=id2label, label2id=label2id)
"""
print(bert_ner_template)

## GPT-style prompting templates

Prompting does not update weights. Use it for MVPs, flexible workflows, and tasks where labels are scarce.

In [ ]:
def build_classification_prompt(text, labels):
    return f"""Classify the customer message into one of these labels: {labels}.
Return only JSON with keys label and rationale.

Message: {text}
"""

print(build_classification_prompt("The app crashes when I upload invoices.", ["bug_report", "billing_issue", "praise", "how_to"]))

## Encoder-decoder template for summarisation or translation

Use T5/BART-style models when the task is naturally text-to-text.

In [ ]:
seq2seq_template = r"""
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"  # or facebook/bart-large-cnn for summarisation
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = "summarize: " + long_document
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
outputs = model.generate(**inputs, max_new_tokens=120, num_beams=4)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)
"""
print(seq2seq_template)

## Product note

Use pipelines to prove value fast. Move to explicit model code when you need:

- repeatable evaluation
- cost/latency control
- batch inference
- data privacy controls
- model/version governance